# Polarized Trees — How the PEG Formulations Change the Trees

Third notebook of the synthetic workflow: `datasetdemo.ipynb` (data) → `treesbenchmark.ipynb` (search, saves the top 20) → **this notebook**.

We take settings from the top 20, keep **everything fixed except the PEG formulation** (`max`, `avg`, `min`, `mean`, `harmonic`), and look at what the formulation changes:

1. **Overview**: recovery and tree shape (leaves, depth, leaf size, residual polarization) for each formulation.
2. **By k**: the same numbers split by the true number of active dimensions of the text, where the formulations differ most.
3. **ARI**: do two formulations split the annotators of a text into the same groups?
4. **F, C, P** on the unseen corpus.
5. **The same text, different trees**: the texts on which the formulations disagree most.

> Run `datasetdemo.ipynb` and `treesbenchmark.ipynb` first, so that `benchmark_data/` and `benchmark_results/top_configurations.csv` exist.

## 1. Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd


# Locate the benchmarks folder (it holds benchmark_config.py).
CWD = Path.cwd().resolve()

if (CWD / "benchmark_config.py").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "benchmark_config.py").exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError(
        "Could not locate project root. "
        "Run this notebook from the notebooks directory "
        "or the project root."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from benchmark_config import (
    DIMS,
    INFERENCE_CORPUS,
    RESULTS_DIR,
    SCALE,
    check_polartox,
    load_benchmark_corpora,
    load_corpus,
)

check_polartox()    # which polartox and which library versions produce these results

from polartox import PolarizedTreesPipeline, pairwise_ari
from polartox.benchmark import STAT_METRICS, STATS
from polartox.polarized_tree import PEG_VARIANTS

# Everything this notebook saves goes here, never over the benchmark notebook's results.
OUT_DIR = RESULTS_DIR / "peg_comparison"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# A, B, C have ground truth; the unseen corpus is used without it.
corpora = load_benchmark_corpora()
unseen, _ = load_corpus(INFERENCE_CORPUS)
datasets = {**corpora, INFERENCE_CORPUS: (unseen, None)}

for name, (data, truth) in datasets.items():
    print(f"{name}: {len(data):,} annotations, {data['text_id'].nunique()} texts"
          f"{'' if truth is not None else ' (no ground truth used)'}")

polartox 0.7.0 imported from C:\Users\swkra\OneDrive\Υπολογιστής\ερευνα\HumanModelAlign\polarizedtrees\polartox\__init__.py
PEG formulations: ('max', 'avg', 'min', 'mean', 'harmonic')
numpy 2.5.3
pandas 3.0.6
ndfu 0.9.3
scikit-learn 1.9.1
A_default: 162,000 annotations, 100 texts
B_weak_signal: 162,000 annotations, 100 texts
C_deep: 162,000 annotations, 100 texts
inference_unseen: 162,000 annotations, 100 texts (no ground truth used)


## 2. Choose the settings

`top_configurations.csv` holds the 20 best configurations of the search. Put the row positions you want in `SELECTED`: each chosen row gives the non-PEG settings, and its own `variant` is **ignored** (every formulation is run with those settings). Choose several rows to compare the formulations at several settings.

In [2]:
top = pd.read_csv(RESULTS_DIR / "top_configurations.csv")

NON_PEG = ["theta_filter", "min_size_frac", "max_depth", "h", "relative_h", "theta_stop"]
display(top[["rank", "variant", *NON_PEG, "jaccard", "exact_match"]].round(4))

SELECTED = [0]      # row positions in the table above

,rank,variant,theta_filter,min_size_frac,max_depth,h,relative_h,theta_stop,jaccard,exact_match
0,1,harmonic,0.3,0.02,4,0.20,True,0.1,0.8924,0.7402
1,2,harmonic,0.3,0.03,6,0.20,True,0.1,0.8924,0.7402
2,3,harmonic,0.3,0.02,6,0.20,True,0.1,0.8924,0.7402
3,4,harmonic,0.3,0.02,4,0.15,True,0.1,0.8875,0.7278
4,5,harmonic,0.3,0.03,4,0.15,True,0.1,0.8875,0.7278
5,6,harmonic,0.2,0.03,4,0.15,True,0.1,0.8845,0.7233
6,7,harmonic,0.4,0.03,4,0.20,True,0.1,0.8832,0.7102
7,8,harmonic,0.4,0.03,6,0.20,True,0.1,0.8832,0.7102
8,9,avg,0.3,0.02,6,0.10,True,0.1,0.8794,0.7096
9,10,avg,0.3,0.02,8,0.10,True,0.1,0.8794,0.7096


## 3. Run every formulation on every corpus

One pipeline per (chosen row, formulation, corpus); `beta` keeps its default of 1.

In [3]:
def make_pipeline(row, variant):
    """The row's non-PEG settings, with the PEG formulation replaced by `variant`."""
    return PolarizedTreesPipeline(
        dims=DIMS,
        scale=SCALE,
        theta_filter=float(row["theta_filter"]),
        min_size_frac=float(row["min_size_frac"]),
        max_depth=int(row["max_depth"]),
        h=float(row["h"]),
        relative_h=bool(row["relative_h"]),
        theta_stop=float(row["theta_stop"]),
        variant=variant,
    )


runs = {}       # (row, formulation, corpus) -> (pipeline, results)
for position in SELECTED:
    for variant in PEG_VARIANTS:
        for corpus, (data, truth) in datasets.items():
            pipe = make_pipeline(top.loc[position], variant)
            results = pipe.run_full_evaluation(data, ground_truth=truth, verbose=False)
            runs[position, variant, corpus] = (pipe, results)

## 4. Overview

One row per formulation, everything else fixed.

- **Recovery** (A/B/C only, where the truth is known): mean and median Jaccard, precision, recall, exact match.
- **Trees**: *Retention* is the share of texts polarized enough to be analysed, *Leaves* and *Depth* are the means per tree (depth is the mean over leaves), *Annotators per leaf* is the mean number of annotators in a leaf (not the number of leaves, which is *Leaves*), taken within each tree and then averaged over the trees, *Residual nDFU* is the polarization left in the leaves, *Top-split PEG* is the mean PEG of the first split, and *Indeterminate* is the share of leaves with no majority pole.

The A/B/C numbers are averaged with equal weight per corpus. The unseen corpus has no ground truth, so only the trees are shown.

In [4]:
RECOVERY = ["jaccard", "precision", "recall", "exact_match"]
TREE = {
    "retention_rate": "Retention", "mean_leaves": "Leaves", "mean_depth": "Depth",
    "mean_leaf_size": "Annotators per leaf", "mean_residual_ndfu": "Residual nDFU",
    "mean_top_split_peg": "Top-split PEG", "indeterminate_rate": "Indeterminate",
}

rows = []
for (position, variant, corpus), (pipe, results) in runs.items():
    leaf_size_per_tree = [np.mean([leaf["n"] for leaf in tree.get_leaves()]) for tree in pipe.trees_.values()]
    row = {
        "top_row": position, "variant": variant, "corpus": corpus,
        **{key: results["diagnostics"][key] for key in TREE if key != "mean_leaf_size"},
        "mean_leaf_size": float(np.mean(leaf_size_per_tree)),
    }
    if "recovery" in results:
        for metric in RECOVERY:
            values = results["recovery"][metric].astype(float)
            row[metric] = values.mean()
            if metric in STAT_METRICS:      # the benchmark's own statistics: median, std, quartiles, min, max
                row.update({f"{metric}_{stat}": function(values) for stat, function in STATS.items()})
    rows.append(row)

per_run = pd.DataFrame(rows)

# A/B/C: every number is the average of the three corpora's values.
abc = per_run[per_run["corpus"].isin(corpora)]
numbers = [c for c in abc.columns if c not in ("top_row", "variant", "corpus")]
overview = abc.groupby(["top_row", "variant"], sort=False)[numbers].mean()

# Sanity check: with its own formulation, a chosen row must reproduce what the benchmark saved.
for position in SELECTED:
    own = top.loc[position, "variant"]
    compared = [c for c in numbers if c in top.columns]
    wrong = [c for c in compared if not np.isclose(overview.loc[(position, own), c], top.loc[position, c])]
    assert not wrong, f"row {position} ({own}) differs from the benchmark in: {wrong}"
print(f"OK: the chosen rows reproduce all {len(compared)} numbers the benchmark saved for them.\n")

SHOWN = {"jaccard": "Jaccard", "jaccard_median": "Jaccard median", "precision": "Precision",
         "recall": "Recall", "exact_match": "Exact match", **TREE}
for position in SELECTED:
    print(f"=== top row {position}: mean over A/B/C ===")
    display(overview.loc[position, list(SHOWN)].rename(columns=SHOWN).rename_axis(None).round(3))

    print(f"=== top row {position}: unseen corpus (no ground truth) ===")
    table = per_run[(per_run["top_row"] == position) & (per_run["corpus"] == INFERENCE_CORPUS)]
    display(table.set_index("variant")[list(TREE)].rename(columns=TREE).rename_axis(None).round(3))

OK: the chosen rows reproduce all 22 numbers the benchmark saved for them.

=== top row 0: mean over A/B/C ===


,Jaccard,Jaccard median,Precision,Recall,Exact match,Retention,Leaves,Depth,Annotators per leaf,Residual nDFU,Top-split PEG,Indeterminate
max,0.808,1.000,0.859,0.936,0.560,0.857,10.585,2.605,286.574,0.235,0.421,0.014
avg,0.848,1.000,0.972,0.872,0.646,0.857,7.216,2.265,358.840,0.193,0.421,0.003
min,0.812,0.917,0.884,0.928,0.554,0.857,10.491,2.589,268.887,0.228,0.494,0.008
mean,0.828,1.000,0.890,0.938,0.588,0.857,10.154,2.558,272.439,0.220,0.429,0.010
harmonic,0.892,1.000,0.952,0.928,0.740,0.857,8.929,2.445,314.560,0.197,0.417,0.005


=== top row 0: unseen corpus (no ground truth) ===


,Retention,Leaves,Depth,Annotators per leaf,Residual nDFU,Top-split PEG,Indeterminate
max,0.86,8.698,2.484,365.586,0.221,0.463,0.005
avg,0.86,6.267,2.126,398.034,0.184,0.465,0.006
min,0.86,9.140,2.515,329.126,0.213,0.522,0.008
mean,0.86,8.616,2.451,335.811,0.208,0.471,0.008
harmonic,0.86,7.593,2.315,357.667,0.179,0.458,0.006


### By the true number of active dimensions (k)

The overview averages over texts with one, two, three or four true causes of polarization. Split by that number, the formulations behave very differently. Pooled over the texts of A, B and C. *Precision* low means extra dimensions were added (over-splitting), *recall* low means true dimensions were missed (under-splitting).

In [ ]:
recovery_by_k = pd.concat(
    results["recovery"].assign(top_row=position, variant=variant, corpus=corpus)
    for (position, variant, corpus), (pipe, results) in runs.items()
    if "recovery" in results
)

for position in SELECTED:
    table = recovery_by_k[recovery_by_k["top_row"] == position]
    print(f"texts per k (top row {position}):", table[table["variant"] == "max"].groupby("k_true").size().to_dict())
    for metric in ("jaccard", "exact_match", "precision", "recall"):
        print(f"=== top row {position}: mean {metric} by k ===")
        display(table.pivot_table(index="k_true", columns="variant", values=metric)[list(PEG_VARIANTS)]
                .rename_axis(None, axis=1).round(3))

## 5. Do the formulations build the same trees? (ARI)

The leaves of a tree split the annotators of a text into groups. The **adjusted Rand index** compares two such splits of the same annotators: **1** means the same groups, **0** means as different as chance. Below, the mean over the texts that all formulations analysed.

In [5]:
ari = {}        # (row, corpus) -> (mean ARI matrix, ARI per text)
for position in SELECTED:
    for corpus, (data, _) in datasets.items():
        pipes = {variant: runs[position, variant, corpus][0] for variant in PEG_VARIANTS}
        ari[position, corpus] = pairwise_ari(pipes, data)

for position in SELECTED:
    print(f"=== top row {position}: mean ARI over A/B/C ===")
    display((sum(ari[position, c][0] for c in corpora) / len(corpora)).round(3))

    print(f"=== top row {position}: unseen corpus ===")
    display(ari[position, INFERENCE_CORPUS][0].round(3))

=== top row 0: mean ARI over A/B/C ===


,max,avg,min,mean,harmonic
max,1.000,0.717,0.783,0.860,0.800
avg,0.717,1.000,0.757,0.793,0.828
min,0.783,0.757,1.000,0.871,0.820
mean,0.860,0.793,0.871,1.000,0.884
harmonic,0.800,0.828,0.820,0.884,1.000


=== top row 0: unseen corpus ===


,max,avg,min,mean,harmonic
max,1.000,0.786,0.842,0.916,0.855
avg,0.786,1.000,0.812,0.835,0.863
min,0.842,0.812,1.000,0.885,0.874
mean,0.916,0.835,0.885,1.000,0.900
harmonic,0.855,0.863,0.874,0.900,1.000


Does the disagreement depend on how many dimensions are really active? Mean ARI by the true number of active dimensions, for every pair of formulations (A/B/C pooled):

In [ ]:
k_of = {corpus: {t: len(truth[t]["active_dims"]) for t in truth} for corpus, (data, truth) in corpora.items()}

ari_by_k = pd.concat(
    ari[position, corpus][1].assign(top_row=position, corpus=corpus, k_true=lambda d, c=corpus: d["text_id"].map(k_of[c]))
    for position in SELECTED
    for corpus in corpora
)
ari_by_k["pair"] = ari_by_k["a"] + " - " + ari_by_k["b"]

for position in SELECTED:
    table = ari_by_k[ari_by_k["top_row"] == position]
    by_pair = table.pivot_table(index="pair", columns="k_true", values="ari")
    by_pair.loc["all pairs"] = table.groupby("k_true")["ari"].mean()
    print(f"=== top row {position}: mean ARI by k ===")
    display(by_pair.rename_axis(None, axis=1).rename_axis(None).round(3))

## 6. Which dimensions and subgroups? (F, C, P)

On the unseen corpus, for every formulation: **F** is the dimension used for the splits at each depth, **C** how consistently a subgroup lands on one pole (5 most frequent), **P** the mean PEG of the split that creates a subgroup (5 highest).

In [6]:
for (position, variant, corpus), (pipe, results) in runs.items():
    if corpus != INFERENCE_CORPUS:
        continue
    print(f"##### top row {position} | {variant}")
    display(results["F"])
    display(results["C"].head(5))
    display(results["P"].head(5))

##### top row 0 | max


depth,1,2,3
dim,,,
age,13,27,28
education,19,24,31
gender,11,23,39
orientation,26,26,22
politics,15,19,45


,n_s,frac_toxic,frac_civil
subgroup,,,
"((education, low),)",13,0.461538,0.538462
"((education, high),)",12,0.500000,0.500000
"((orientation, lgbtq+),)",11,0.545455,0.454545
"((orientation, heterosexual),)",11,0.545455,0.454545
"((education, medium),)",11,0.727273,0.272727


,n_s,mean_peg
subgroup,,
"((gender, male), (politics, center))",1,0.716268
"((gender, male), (politics, right))",1,0.716268
"((education, low), (politics, left))",1,0.696192
"((age, <25), (politics, right))",1,0.676224
"((orientation, lgbtq+),)",11,0.675417


##### top row 0 | avg


...

,n_s,mean_peg
subgroup,,
"((education, low), (gender, non-binary), (politics, center))",1,0.714231
"((education, high), (gender, non-binary), (politics, left))",1,0.714105
"((education, high), (gender, non-binary), (politics, center))",1,0.714105
"((education, low), (orientation, heterosexual), (politics, center))",1,0.701121
"((age, 25-50), (gender, non-binary), (orientation, heterosexual))",1,0.695148


##### top row 0 | harmonic


depth,1,2,3
dim,,,
age,16,21,23
education,17,29,19
gender,15,21,20
orientation,25,25,27
politics,13,25,26


,n_s,frac_toxic,frac_civil
subgroup,,,
"((education, low),)",13,0.461538,0.538462
"((orientation, heterosexual),)",12,0.583333,0.416667
"((education, high),)",12,0.500000,0.500000
"((orientation, lgbtq+),)",12,0.500000,0.500000
"((education, medium),)",11,0.727273,0.272727


,n_s,mean_peg
subgroup,,
"((age, >50), (education, high), (politics, left))",1,0.721927
"((age, >50), (gender, male), (orientation, lgbtq+))",2,0.707905
"((education, low), (gender, non-binary), (politics, right))",1,0.686892
"((education, low), (gender, non-binary), (politics, center))",1,0.686892
"((education, low), (orientation, lgbtq+), (politics, center))",1,0.678253


## 7. The same text, different trees

Where do the formulations disagree most? For every text of A/B/C we take the mean ARI over all pairs of formulations; the lowest values are the texts on which the formulations build the most different trees (for the first chosen row). For each of them: the true structure, one line per formulation, and the trees themselves.

Set `TEXTS = [("A_default", 12), ...]` to look at specific texts instead.

In [ ]:
N_TEXTS = 3
TEXTS = None        # or a list of (corpus, text_id)

disagreement = pd.concat(
    ari[SELECTED[0], corpus][1].groupby("text_id")["ari"].mean().rename("mean_ari").reset_index().assign(corpus=corpus)
    for corpus in corpora
).sort_values("mean_ari")
display(disagreement.head(N_TEXTS).reset_index(drop=True))

chosen = TEXTS or list(zip(disagreement["corpus"].head(N_TEXTS), disagreement["text_id"].head(N_TEXTS)))


def show_text(position, corpus, text_id):
    data, truth = datasets[corpus]
    true_dims = truth[text_id]["active_dims"]
    alpha = truth[text_id]["alpha"]
    print("#" * 10, f"{corpus}, text {text_id} | true active dimensions:",
          ", ".join(f"{d} (alpha {alpha[d]:.2f})" for d in true_dims))

    rows, trees = [], {}
    for variant in PEG_VARIANTS:
        tree = runs[position, variant, corpus][0].trees_.get(text_id)
        if tree is None:
            continue
        used = sorted({dim for leaf in tree.get_leaves() for dim, _ in leaf["path"]})
        rows.append({
            "formulation": variant,
            "first split": tree.get_root().get("split_dim", "none"),
            "dimensions used": ", ".join(used) or "none",
            "leaves": tree.n_leaves,
            "depth": tree.depth,
            "Jaccard": len(set(used) & set(true_dims)) / len(set(used) | set(true_dims)),
        })
        trees[variant] = tree
    display(pd.DataFrame(rows).set_index("formulation").round(2))

    for variant, tree in trees.items():
        print(f"--- {variant} ---")
        tree.render()


for corpus, text_id in chosen:
    show_text(SELECTED[0], corpus, text_id)

## 8. Save

Everything goes into `benchmark_results/peg_comparison/`: `per_run.csv` has every number of every run and corpus, `overview_A_B_C.csv` the table above, `recovery_by_k.csv` and `ari_by_k.csv` the by-k tables, the ARI matrices and per-text values, and the complete F, C and P tables.

In [ ]:
top.loc[SELECTED].to_csv(OUT_DIR / "selected_settings.csv")
per_run.to_csv(OUT_DIR / "per_run.csv", index=False)
overview.to_csv(OUT_DIR / "overview_A_B_C.csv")
recovery_by_k.groupby(["top_row", "variant", "k_true"]).mean(numeric_only=True).to_csv(OUT_DIR / "recovery_by_k.csv")
ari_by_k.groupby(["top_row", "pair", "k_true"])["ari"].mean().to_csv(OUT_DIR / "ari_by_k.csv")

for (position, corpus), (matrix, per_text) in ari.items():
    matrix.to_csv(OUT_DIR / f"ari_top{position}_{corpus}.csv")
    per_text.to_csv(OUT_DIR / f"ari_per_text_top{position}_{corpus}.csv", index=False)

for (position, variant, corpus), (pipe, results) in runs.items():
    if corpus == INFERENCE_CORPUS:
        for name in "FCP":
            results[name].to_csv(OUT_DIR / f"{name}_top{position}_{variant}.csv")

print("Saved to:", OUT_DIR.resolve())